# Understanding SIFT: Scale-Invariant Feature Transform

*A comprehensive guide to implementing SIFT from scratch*

---

The **Scale-Invariant Feature Transform (SIFT)** is one of the most influential algorithms in computer vision. Introduced by David Lowe in 2004, SIFT detects and describes local features in images that remain stable across changes in scale, rotation, and illumination.

This notebook walks through the complete SIFT pipeline, from mathematical foundations to practical implementation.

## Table of Contents

1. [Overview](#overview)
2. [Detection Phase](#detection-phase)
   - Step 1: Gaussian Scale-Space Pyramid
   - Step 2: Difference of Gaussians
   - Step 3: Keypoint Detection
   - Step 4: Keypoint Filtering & Refinement
3. [Description Phase](#description-phase)
   - Step 5: Orientation Assignment
   - Step 6: Descriptor Extraction
4. [Summary](#summary)

## Overview

SIFT operates in two main phases:

| Phase | Step | Description | Math |
|-------|------|-------------|------|
| Detection | 1 | Gaussian pyramid | `L(x,y,σ) = G(x,y,σ) * I(x,y)` |
| Detection | 2 | DoG | `D = L(kσ) - L(σ)` |
| Detection | 3 | Keypoint detection | 26-neighbor extrema |
| Detection | 4 | Refinement & Filtering | Taylor expansion + edge removal |
| Description | 5 | Orientation | 36-bin histogram |
| Description | 6 | Descriptor | 128-D |

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.ndimage import gaussian_filter
from PIL import Image
from IPython.display import display, Markdown
import os

# For better plots
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print("Setup complete!")

## Load Input Image

In [ ]:
# Load the input image
img_path = 'images/input_image.jpg'
img = np.array(Image.open(img_path).convert('L'))
H, W = img.shape

plt.figure(figsize=(10, 8))
plt.imshow(img, cmap='gray')
plt.title(f'Input Image: {W} × {H} pixels', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Image shape: {img.shape}")

---
# 1️⃣ DETECTION PHASE

**Goal**: Find stable, repeatable keypoints in the image that can be detected regardless of scale, rotation, or illumination changes.

```
INPUT: Image (H × W)
        ↓
Step 1: Build Gaussian Scale-Space Pyramid
        ↓
Step 2: Compute Difference of Gaussians (DoG)
        ↓
Step 3: Detect Keypoints (26-neighbor extrema) → 1124 keypoints
        ↓
Step 4: Filter & Refine Keypoints
        - Low Contrast Removal     → 1124 keypoints
        - Edge Response Removal    → 963 keypoints
        - Sub-pixel Refinement     → 847 keypoints
        ↓
OUTPUT: 847 stable keypoints with (x, y, scale)
```

---
## Step 1: Gaussian Scale-Space Pyramid

### Why do we need this?

To detect features at any scale, we must analyze the image at multiple resolutions.

### The Mathematics

The scale-space representation is created by convolving the image with Gaussian kernels of increasing width:

**Gaussian Blur:**
$$L(x,y,\sigma) = G(x,y,\sigma) * I(x,y)$$

**Gaussian Kernel:**
$$G(x,y,\sigma) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}}$$

Where:
- $L(x,y,\sigma)$ is the scale-space representation
- $G(x,y,\sigma)$ is the Gaussian kernel
- $I(x,y)$ is the input image
- $\sigma$ is the scale parameter

**Scale Space:**
- σ increases by factor k = √2 at each scale
- Multiple octaves (image resolutions)
- Each octave has multiple scales

### Step 1 Visualization

![Step 1: Gaussian Scale-Space Pyramid](images/sift_step1_gaussian_pyramid.png)

In [ ]:
def build_gaussian_pyramid(img, n_octaves=3, n_scales=5, sigma=1.6, k=np.sqrt(2)):
    """
    Build Gaussian Scale-Space Pyramid.
    
    Parameters:
    - img: Input grayscale image
    - n_octaves: Number of octaves (resolution levels)
    - n_scales: Number of scales per octave
    - sigma: Base sigma value
    - k: Scale factor between consecutive scales (√2)
    
    Returns:
    - pyramid: List of octaves, each containing list of blurred images
    
    Math:
    σ_scale = σ_base × k^scale
    """
    pyramid = []
    current = img.astype(np.float64)
    
    for octave in range(n_octaves):
        octave_imgs = []
        for scale in range(n_scales):
            sig = sigma * (k ** scale)
            blurred = gaussian_filter(current, sigma=sig)
            octave_imgs.append(blurred)
        pyramid.append(octave_imgs)
        current = current[::2, ::2]  # Downsample for next octave
    
    return pyramid

# Build pyramid
pyramid = build_gaussian_pyramid(img)

print(f"Pyramid built with {len(pyramid)} octaves")
for i, octave in enumerate(pyramid):
    print(f"  Octave {i}: {len(octave)} scales, resolution {octave[0].shape[1]}×{octave[0].shape[0]}")

---
## Step 2: Difference of Gaussians (DoG)

### Why DoG?

The Difference of Gaussians approximates the **Laplacian of Gaussian (LoG)**, which is an excellent blob detector.

### The Mathematics

**Difference of Gaussians:**
$$D(x,y,\sigma) = L(x,y,k\sigma) - L(x,y,\sigma)$$

**Approximation:**
$$D(x,y,\sigma) \approx (k-1)\sigma^2 \nabla^2 G * I$$

By subtracting adjacent scales, we get a response that highlights blob-like structures at different sizes.

This is computationally cheaper than computing the actual Laplacian.

### Step 2 Visualization

![Step 2: Difference of Gaussians](images/sift_step2_dog.png)

In [ ]:
def compute_dog(pyramid):
    """
    Compute Difference of Gaussians from Gaussian pyramid.
    
    Math:
    D(x,y,σ) = L(x,y,kσ) - L(x,y,σ)
    
    Parameters:
    - pyramid: Gaussian scale-space pyramid
    
    Returns:
    - dog_pyramid: DoG pyramid (one less scale per octave)
    """
    dog_pyramid = []
    
    for octave_imgs in pyramid:
        dog_octave = []
        for i in range(len(octave_imgs) - 1):
            dog = octave_imgs[i + 1] - octave_imgs[i]
            dog_octave.append(dog)
        dog_pyramid.append(dog_octave)
    
    return dog_pyramid

# Compute DoG
dog_pyramid = compute_dog(pyramid)

print(f"DoG pyramid built with {len(dog_pyramid)} octaves")
for i, octave in enumerate(dog_pyramid):
    print(f"  Octave {i}: {len(octave)} DoG images")

---
## Step 3: Keypoint Detection (26-Neighbor Extrema)

Keypoint detection uses **26-neighbor comparison** across three consecutive DoG images.

### Step 3.1: Three DoG Scales

We need three consecutive DoG images for scale-space extrema detection:

![Step 3.1: Three Scales](images/sift_step3_1_three_scales.png)

### Step 3.2: Understanding the 26 Neighbors

For each pixel at (x, y, σ), compare with **26 neighbors**:
- 9 at scale σ-1 (previous)
- 8 at scale σ (same scale, exclude center)
- 9 at scale σ+1 (next)

```
Keypoint if:
  value > ALL 26 neighbors → Maximum
  value < ALL 26 neighbors → Minimum
```

![Step 3.2: 26 Neighbors](images/sift_step3_2_26_neighbors.png)

### Step 3.3-3.5: Multi-Octave Processing

The process repeats at multiple octaves (resolutions):

| Octave | Resolution | Processing |
|--------|------------|------------|
| 0 | H × W | Full resolution keypoints |
| 1 | H/2 × W/2 | Half resolution, scaled back 2× |
| 2 | H/4 × W/4 | Quarter resolution, scaled back 4× |

![Step 3.3: Octave 0](images/sift_step3_3_octave0.png)

![Step 3.4: Octave 1](images/sift_step3_4_octave1.png)

![Step 3.5: Octave 2](images/sift_step3_5_octave2.png)

### Step 3.6: Complete Pyramid Structure

```
OCTAVE 0 (H×W):      G(σ₁) → G(σ₂) → G(σ₃) → G(σ₄)  →  DoG → 26-nbr → KP
    ↓ downsample
OCTAVE 1 (H/2×W/2):  G(σ₁) → G(σ₂) → G(σ₃) → G(σ₄)  →  DoG → 26-nbr → KP
    ↓ downsample
OCTAVE 2 (H/4×W/4):  G(σ₁) → G(σ₂) → G(σ₃) → G(σ₄)  →  DoG → 26-nbr → KP
```

![Step 3.6: Pyramid Structure](images/sift_step3_6_pyramid_structure.png)

### Coordinate Transformation Between Octaves

Keypoints from different octaves must be scaled back to the original image coordinates:

| Octave | Resolution | Scale Factor | Coordinate Transform |
|--------|------------|--------------|---------------------|
| 0 | H × W | 1 | (x, y) → (x, y) |
| 1 | H/2 × W/2 | 2 | (x, y) → (x×2, y×2) |
| 2 | H/4 × W/4 | 4 | (x, y) → (x×4, y×4) |

**Implementation:**

```python
# Octave 0: no scaling
kp0 = detect_keypoints(gray)  # [(x, y, type), ...]

# Octave 1: scale by 2
gray1 = gray[::2, ::2]  # Downsample
kp1_local = detect_keypoints(gray1)
kp1 = [(x*2, y*2, t) for x, y, t in kp1_local]  # Scale back

# Octave 2: scale by 4
gray2 = gray[::4, ::4]  # Downsample
kp2_local = detect_keypoints(gray2)
kp2 = [(x*4, y*4, t) for x, y, t in kp2_local]  # Scale back

# Combine all
all_keypoints = kp0 + kp1 + kp2
```

![Scale Factor Visualization](images/sift_scale_factor_real.png)

### All Octaves Combined

Circle size and color indicate detection scale:
- **Red small circles**: Octave 0 (Fine-scale) - 856 keypoints
- **Green medium circles**: Octave 1 (Medium-scale) - 213 keypoints
- **Blue large circles**: Octave 2 (Coarse-scale) - 55 keypoints

![All Octaves Combined](images/sift_all_octaves_combined.png)

```
OCTAVE 0 (H × W):      → 856 keypoints (Red, Small)
OCTAVE 1 (H/2 × W/2):  → 213 keypoints (Green, Medium)
OCTAVE 2 (H/4 × W/4):  →  55 keypoints (Blue, Large)
                       ─────────────
TOTAL DETECTED:        1124 keypoints
```

In [ ]:
def detect_extrema(dog_pyramid):
    """
    Detect scale-space extrema using 26-neighbor comparison.
    
    For each pixel at (x, y) in the middle DoG:
    - Compare to 8 neighbors at same scale
    - Compare to 9 neighbors at scale above
    - Compare to 9 neighbors at scale below
    - Total: 26 neighbors
    
    Returns:
    - keypoints: List of detected keypoints
    """
    keypoints = []
    keypoints_per_octave = []
    
    for octave_idx, dog_octave in enumerate(dog_pyramid):
        scale_factor = 2 ** octave_idx
        octave_kps = []
        
        for scale_idx in range(1, len(dog_octave) - 1):
            prev_dog = dog_octave[scale_idx - 1]
            curr_dog = dog_octave[scale_idx]
            next_dog = dog_octave[scale_idx + 1]
            
            h, w = curr_dog.shape
            
            for y in range(1, h - 1):
                for x in range(1, w - 1):
                    val = curr_dog[y, x]
                    
                    neighbors = []
                    for dy in [-1, 0, 1]:
                        for dx in [-1, 0, 1]:
                            neighbors.append(prev_dog[y + dy, x + dx])
                            if dy != 0 or dx != 0:
                                neighbors.append(curr_dog[y + dy, x + dx])
                            neighbors.append(next_dog[y + dy, x + dx])
                    
                    if val > max(neighbors) or val < min(neighbors):
                        kp = {
                            'x': x * scale_factor,
                            'y': y * scale_factor,
                            'local_x': x,
                            'local_y': y,
                            'octave': octave_idx,
                            'scale': scale_idx,
                            'response': abs(val)
                        }
                        keypoints.append(kp)
                        octave_kps.append(kp)
        
        keypoints_per_octave.append(octave_kps)
    
    return keypoints, keypoints_per_octave

# Detect keypoints
keypoints, keypoints_per_octave = detect_extrema(dog_pyramid)

print(f"Total keypoints detected: {len(keypoints)}")
for i, kps in enumerate(keypoints_per_octave):
    print(f"  Octave {i}: {len(kps)} keypoints")

---
## Step 4: Keypoint Filtering & Refinement

The initial keypoints are filtered through three stages:

```
Step 3 Complete: 1124 keypoints detected
        ↓
Stage 1: Low Contrast Removal     |D(x̂)| < 0.03      → 1124 remaining
        ↓
Stage 2: Edge Response Removal    Tr(H)²/Det(H) > 12.1 → 963 remaining
        ↓
Stage 3: Sub-pixel Refinement     |offset| > 0.5      → 847 remaining
        ↓
FINAL: 847 stable keypoints (75.4% of 1124)
```

### Understanding the Filtering Coordinates

**(x, y) refers to keypoint coordinates, NOT all image pixels.**

```
Image size: 640 × 480 = 307,200 total pixels

Step 3 detected: 1124 keypoints (blob points)
  - Each keypoint has coordinates (x, y) where it was detected

Filtering is applied ONLY to these 1124 points:
  - For each keypoint:
    1. Look at 3×3 neighborhood around (x, y) in DoG image
    2. Compute derivatives using neighboring pixels
    3. Apply filtering tests
    4. Keep or reject
```

### Why a 3×3 Window?

We need a **3×3 window** around each keypoint to compute derivatives using **finite differences**:

```
Keypoint at (x, y):

       x-1    x    x+1
      ┌─────┬─────┬─────┐
y-1   │ NW  │  N  │ NE  │   ← Need y-1 for Dy, Dyy, Dxy
      ├─────┼─────┼─────┤
y     │  W  │  C  │  E  │   ← Need x-1, x+1 for Dx, Dxx
      ├─────┼─────┼─────┤
y+1   │ SW  │  S  │ SE  │   ← Need y+1 for Dy, Dyy, Dxy
      └─────┴─────┴─────┘

C = Center (keypoint location)
```

### Derivative Computations

**First Derivatives (Gradient):**

$$D_x = \frac{D(x+1,y) - D(x-1,y)}{2}$$

$$D_y = \frac{D(x,y+1) - D(x,y-1)}{2}$$

**Second Derivatives (Curvature):**

$$D_{xx} = D(x+1,y) + D(x-1,y) - 2 \times D(x,y)$$

$$D_{yy} = D(x,y+1) + D(x,y-1) - 2 \times D(x,y)$$

$$D_{xy} = \frac{D(x+1,y+1) - D(x+1,y-1) - D(x-1,y+1) + D(x-1,y-1)}{4}$$

### Stage 1: Low Contrast Removal

**Purpose**: Remove keypoints sensitive to noise.

**Mathematics:**

Compute contrast at refined location using Taylor expansion:

$$D(\hat{x}) = D + \frac{1}{2} \nabla D^T \cdot \text{offset}$$

**Reject if:** $|D(\hat{x})| < 0.03$

**Example - Keypoint KEPT:**

```
DoG neighborhood around (50, 80):
       x=49   x=50   x=51
      ┌──────┬──────┬──────┐
y=79  │  15  │  18  │  12  │
      ├──────┼──────┼──────┤
y=80  │  20  │  45  │  25  │  ← Center = 45 (strong response)
      ├──────┼──────┼──────┤
y=81  │  14  │  22  │  16  │
      └──────┴──────┴──────┘

D(x̂) = 45.112
|45.112| < 0.03? NO → KEEP
```

![Stage 1: Low Contrast](images/sift_stage1_low_contrast.png)

### Stage 2: Edge Response Removal

**Purpose**: Remove keypoints on edges (poorly localized along edge direction).

**Mathematics:**

Compute Hessian matrix:
$$H = \begin{bmatrix} D_{xx} & D_{xy} \\ D_{xy} & D_{yy} \end{bmatrix}$$

Edge ratio test:
$$\frac{Tr(H)^2}{Det(H)} = \frac{(D_{xx} + D_{yy})^2}{D_{xx} \cdot D_{yy} - D_{xy}^2}$$

**Reject if:** $\frac{Tr(H)^2}{Det(H)} > \frac{(r+1)^2}{r}$ where $r = 10$

Threshold = $(10+1)^2 / 10 = 12.1$

**Example - Edge Rejected:**

```
Keypoint at (350, 200):
Strong response along Y-axis but weak along X-axis → EDGE

Dxx = -59, Dyy = -3, Dxy = 0
Tr(H) = -62
Det(H) = 177
Ratio = (-62)² / 177 = 21.72 > 12.1 → REJECT
```

![Stage 2: Edge Response](images/sift_stage2_edge_response.png)

### Stage 3: Sub-pixel Refinement

**Purpose**: Remove unstable keypoints where the true extremum is in a different pixel.

**Mathematics:**

Compute sub-pixel offset:
$$\text{offset} = -H^{-1} \cdot \nabla D$$

**Reject if:** $|\text{offset}_x| > 0.5$ OR $|\text{offset}_y| > 0.5$

**Example - Unstable Rejected:**

```
Keypoint at (80, 420):
offset_x = -0.86  ← Too large! True peak almost 1 pixel away.

RESULT: REJECTED - Unstable keypoint
```

![Stage 3: Subpixel](images/sift_stage3_subpixel.png)

In [ ]:
def compute_derivatives(dog, x, y):
    """
    Compute first and second derivatives using finite differences.
    
    Uses 3×3 neighborhood around (x, y).
    """
    # First derivatives (gradient)
    dx = (dog[y, x + 1] - dog[y, x - 1]) / 2.0
    dy = (dog[y + 1, x] - dog[y - 1, x]) / 2.0
    
    # Second derivatives (Hessian)
    dxx = dog[y, x + 1] + dog[y, x - 1] - 2 * dog[y, x]
    dyy = dog[y + 1, x] + dog[y - 1, x] - 2 * dog[y, x]
    dxy = (dog[y + 1, x + 1] - dog[y + 1, x - 1] - 
           dog[y - 1, x + 1] + dog[y - 1, x - 1]) / 4.0
    
    return dx, dy, dxx, dyy, dxy

def filter_keypoints(keypoints, dog_pyramid, contrast_threshold=0.03, edge_threshold=10):
    """
    Apply all three filtering stages.
    
    Stage 1: Low contrast removal  |D(x̂)| < 0.03
    Stage 2: Edge response removal Tr²/Det > 12.1
    Stage 3: Sub-pixel refinement  |offset| > 0.5
    """
    stages = {
        'detected': len(keypoints),
        'after_contrast': 0,
        'after_edge': 0,
        'after_subpixel': 0
    }
    
    edge_limit = ((edge_threshold + 1) ** 2) / edge_threshold  # 12.1
    
    after_contrast = []
    after_edge = []
    filtered = []
    
    for kp in keypoints:
        octave = kp['octave']
        scale = kp['scale']
        local_x = kp['local_x']
        local_y = kp['local_y']
        
        dog = dog_pyramid[octave][scale]
        h, w = dog.shape
        
        if not (1 <= local_x < w - 1 and 1 <= local_y < h - 1):
            continue
        
        dx, dy, dxx, dyy, dxy = compute_derivatives(dog, local_x, local_y)
        
        det_h = dxx * dyy - dxy ** 2
        if det_h == 0:
            continue
        
        # Stage 1: Low Contrast
        offset_x = -(dyy * dx - dxy * dy) / det_h
        offset_y = -(dxx * dy - dxy * dx) / det_h
        D_refined = dog[local_y, local_x] + 0.5 * (dx * offset_x + dy * offset_y)
        
        if abs(D_refined) < contrast_threshold:
            continue
        after_contrast.append(kp)
        
        # Stage 2: Edge Response
        tr_h = dxx + dyy
        if det_h <= 0:
            continue
        edge_ratio = (tr_h ** 2) / det_h
        if edge_ratio > edge_limit:
            continue
        after_edge.append(kp)
        
        # Stage 3: Sub-pixel Refinement
        if abs(offset_x) > 0.5 or abs(offset_y) > 0.5:
            continue
        
        kp['refined_x'] = kp['x'] + offset_x * (2 ** octave)
        kp['refined_y'] = kp['y'] + offset_y * (2 ** octave)
        filtered.append(kp)
    
    stages['after_contrast'] = len(after_contrast)
    stages['after_edge'] = len(after_edge)
    stages['after_subpixel'] = len(filtered)
    
    return filtered, stages, after_contrast, after_edge

# Filter keypoints
filtered_keypoints, stages, after_contrast, after_edge = filter_keypoints(keypoints, dog_pyramid)

print("Filtering Results:")
print(f"  Detected:         {stages['detected']}")
print(f"  After Contrast:   {stages['after_contrast']} (-{stages['detected'] - stages['after_contrast']} low contrast)")
print(f"  After Edge:       {stages['after_edge']} (-{stages['after_contrast'] - stages['after_edge']} edges)")
print(f"  After Sub-pixel:  {stages['after_subpixel']} (-{stages['after_edge'] - stages['after_subpixel']} unstable)")
print(f"  Retention:        {100*stages['after_subpixel']/stages['detected']:.1f}%")

---
# ✅ Detection Phase Complete!

We now have stable keypoints with (x, y, scale).

---
# 2️⃣ DESCRIPTION PHASE

**Goal**: Create unique, rotation-invariant, scale-invariant fingerprints (descriptors) for each detected keypoint.

```
INPUT: 847 stable keypoints with (x, y, scale)
        ↓
Step 5: Orientation Assignment
        - 36-bin gradient histogram around each keypoint
        ↓
Step 6: Descriptor Extraction
        - 16×16 region → 4×4 subregions → 8-bin histograms
        - 128-D vector
        ↓
OUTPUT: 847 keypoints with (x, y, scale, orientation, 128-D descriptor)
```

---
## Step 5: Orientation Assignment

**Purpose**: Achieve rotation invariance by assigning a dominant orientation to each keypoint.

### The Mathematics

For each pixel (x, y) in a region:

**Gradient:**
$$G_x = I(x+1, y) - I(x-1, y)$$
$$G_y = I(x, y+1) - I(x, y-1)$$

**Magnitude:**
$$m(x,y) = \sqrt{G_x^2 + G_y^2}$$

**Direction:**
$$\theta(x,y) = \arctan\left(\frac{G_y}{G_x}\right) \rightarrow 0° \text{ to } 360°$$

**36-bin Histogram:**
- Each bin covers 10°
- Weight = magnitude × Gaussian(distance from center)
- Dominant direction = peak bin

![Orientation Assignment](images/sift_desc_orientation.png)

In [ ]:
def compute_gradients(img):
    """Compute gradient magnitude and orientation for entire image."""
    gy, gx = np.gradient(img.astype(np.float64))
    magnitude = np.sqrt(gx**2 + gy**2)
    orientation = np.arctan2(gy, gx) * 180 / np.pi
    orientation = (orientation + 360) % 360
    return magnitude, orientation, gx, gy

def compute_orientation(magnitude, orientation, kp_x, kp_y, radius=8):
    """Compute dominant orientation using 36-bin histogram."""
    hist = np.zeros(36)
    h, w = magnitude.shape
    
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            y, x = int(kp_y + dy), int(kp_x + dx)
            if 0 <= y < h and 0 <= x < w:
                mag = magnitude[y, x]
                ori = orientation[y, x]
                weight = np.exp(-(dx**2 + dy**2) / (2 * (radius/2)**2))
                bin_idx = int(ori / 10) % 36
                hist[bin_idx] += mag * weight
    
    dominant_bin = np.argmax(hist)
    dominant_orientation = dominant_bin * 10 + 5
    return dominant_orientation, hist

# Compute gradients
magnitude, orientation, gx, gy = compute_gradients(img)

# Assign orientations
for kp in filtered_keypoints:
    dom_ori, hist = compute_orientation(magnitude, orientation, kp['x'], kp['y'])
    kp['orientation'] = dom_ori
    kp['ori_hist'] = hist

print(f"Orientation assigned to {len(filtered_keypoints)} keypoints")

---
## Step 6: Descriptor Extraction (128-D)

**Purpose**: Create a unique 128-dimensional fingerprint for matching across images.

### Descriptor Overview

![SIFT Descriptor Overview](images/sift_descriptor_overview.png)

### Step 6.1: Extract 16×16 Region

Center a 16×16 pixel region on the keypoint, rotated according to the dominant orientation:

```
16×16 Region:
  - Centered on keypoint position (x, y)
  - Rotated by dominant orientation θ
  - Total: 256 pixels
```

![16x16 Region](images/sift_desc_16x16.png)

### Step 6.2: Divide into 4×4 Subregions

```
┌────┬────┬────┬────┐
│ S0 │ S1 │ S2 │ S3 │   Each subregion = 4×4 pixels
├────┼────┼────┼────┤
│ S4 │ S5 │ S6 │ S7 │   Total subregions = 16
├────┼────┼────┼────┤
│ S8 │ S9 │S10 │S11 │   Each subregion → 8-bin histogram
├────┼────┼────┼────┤
│S12 │S13 │S14 │S15 │
└────┴────┴────┴────┘
```

![4x4 Grid](images/sift_desc_4x4grid.png)

### Step 6.3: Compute Gradient Directions

![Gradients](images/sift_desc_gradients.png)

### Step 6.4: Build 8-bin Histogram per Subregion

```
8 bins (45° each):
  Bin 0: 0° - 45°      Bin 4: 180° - 225°
  Bin 1: 45° - 90°     Bin 5: 225° - 270°
  Bin 2: 90° - 135°    Bin 6: 270° - 315°
  Bin 3: 135° - 180°   Bin 7: 315° - 360°

Result: 16 subregions × 8 bins = 128 values
```

![8-bin Histograms](images/sift_desc_histograms.png)

### Step 6.5: Create Final 128-D Descriptor

**Normalization:**
1. L2 normalize: $d = d / ||d||$
2. Clip values > 0.2 (reduce illumination effects)
3. Re-normalize

![128-D Descriptor](images/sift_desc_final128.png)

In [ ]:
def compute_descriptor(magnitude, orientation, kp_x, kp_y, kp_orientation):
    """
    Compute 128-D SIFT descriptor.
    
    Process:
    1. Extract 16×16 region
    2. Divide into 4×4 = 16 subregions
    3. Build 8-bin histogram per subregion
    4. Concatenate and normalize
    """
    h, w = magnitude.shape
    half = 8
    
    y_start = max(0, int(kp_y) - half)
    y_end = min(h, int(kp_y) + half)
    x_start = max(0, int(kp_x) - half)
    x_end = min(w, int(kp_x) + half)
    
    region_mag = magnitude[y_start:y_end, x_start:x_end]
    region_ori = orientation[y_start:y_end, x_start:x_end]
    
    region_ori = (region_ori - kp_orientation + 360) % 360
    
    if region_mag.shape[0] < 16 or region_mag.shape[1] < 16:
        return None, None
    
    region_mag = region_mag[:16, :16]
    region_ori = region_ori[:16, :16]
    
    histograms = []
    for row in range(4):
        for col in range(4):
            hist = np.zeros(8)
            for dy in range(4):
                for dx in range(4):
                    py = row * 4 + dy
                    px = col * 4 + dx
                    mag = region_mag[py, px]
                    ori = region_ori[py, px]
                    bin_idx = int(ori / 45) % 8
                    hist[bin_idx] += mag
            histograms.append(hist)
    
    descriptor = np.concatenate(histograms)
    
    # Normalize
    norm = np.linalg.norm(descriptor)
    if norm > 0:
        descriptor = descriptor / norm
    descriptor = np.clip(descriptor, 0, 0.2)
    norm = np.linalg.norm(descriptor)
    if norm > 0:
        descriptor = descriptor / norm
    
    return descriptor, histograms

# Compute descriptors
for kp in filtered_keypoints:
    desc, hists = compute_descriptor(magnitude, orientation, kp['x'], kp['y'], kp['orientation'])
    kp['descriptor'] = desc
    kp['histograms'] = hists

valid_descriptors = sum(1 for kp in filtered_keypoints if kp['descriptor'] is not None)
print(f"Computed {valid_descriptors} descriptors (128-D each)")

### Complete Descriptor Pipeline

![Descriptor Pipeline](images/sift_desc_pipeline_real.png)

---
# Summary

## Complete SIFT Pipeline

```
INPUT: Image (H × W)

═══════════════════════════════════════════════════════════════════
                        DETECTION PHASE
═══════════════════════════════════════════════════════════════════
        ↓
STEP 1: Gaussian Scale-Space Pyramid
        ↓
STEP 2: Difference of Gaussians (DoG)
        ↓
STEP 3: Keypoint Detection (26-neighbor extrema)
        ↓
STEP 4: Keypoint Filtering & Refinement → 847 keypoints

═══════════════════════════════════════════════════════════════════
                        DESCRIPTION PHASE
═══════════════════════════════════════════════════════════════════
        ↓
STEP 5: Orientation Assignment
        ↓
STEP 6: Descriptor Extraction → 128-D per keypoint
        ↓
OUTPUT: 847 keypoints with (x, y, scale, orientation, 128-D descriptor)
```

![Complete SIFT Pipeline](images/sift_complete_summary.png)

## Quick Reference: Filtering Formulas

| Stage | Formula | Threshold | Action |
|-------|---------|-----------|--------|
| Stage 1 | $D(\hat{x}) = D + 0.5 \times \nabla D \cdot \text{offset}$ | $|D(\hat{x})| < 0.03$ | REJECT |
| Stage 2 | $\frac{Tr(H)^2}{Det(H)}$ | $> 12.1$ | REJECT |
| Stage 3 | $\text{offset} = -H^{-1} \times \nabla D$ | $|\text{offset}| > 0.5$ | REJECT |

## Key Properties

| Property | Value |
|----------|-------|
| Year | 2004 (Lowe) |
| Speed | Slower than SURF |
| Detection | DoG extrema → 1124 keypoints |
| Filtering | 1124 → 847 (75.4% retention) |
| Description | 128-D descriptor per keypoint |
| Key Innovation | Scale-space pyramid + sub-pixel accuracy |

In [ ]:
# Print final statistics
print("="*60)
print("SIFT ALGORITHM COMPLETE")
print("="*60)
print(f"\nInput Image: {W} × {H} pixels")
print(f"\n1️⃣ DETECTION PHASE:")
print(f"   Step 1: Gaussian Pyramid - {len(pyramid)} octaves")
print(f"   Step 2: DoG - {sum(len(o) for o in dog_pyramid)} DoG images")
print(f"   Step 3: Detected - {stages['detected']} keypoints")
print(f"   Step 4: Filtered - {stages['after_subpixel']} keypoints ({100*stages['after_subpixel']/stages['detected']:.1f}% retained)")
print(f"\n2️⃣ DESCRIPTION PHASE:")
print(f"   Step 5: Orientation assigned to all keypoints")
print(f"   Step 6: {valid_descriptors} × 128-D descriptors computed")
print(f"\nFINAL OUTPUT: {valid_descriptors} keypoints with 128-D descriptors")
print("="*60)

---
## References

1. Lowe, D. G. (2004). "Distinctive Image Features from Scale-Invariant Keypoints." International Journal of Computer Vision, 60(2), 91-110.